# Decision Tree

The steps for inducing a decision tree are as follows:
- Start with all (training) examples at the root node
- Calculate information gain for splitting on all possible features, and pick the one with the highest information gain
- Split dataset according to the selected feature, and create left and right branches of the tree
- Keep repeating the splitting process until stopping criteria is met
  


In [13]:
import numpy as np                                       # for computing log2
import matplotlib.pyplot as plt                          # for one plot only

import pandas as pd                                      # for data manipulation

from sklearn.model_selection import train_test_split     # for splitting the data

In [ ]:
from pathlib import Path

data_dir = Path("../../data")
data = pd.read_csv(data_dir / "zoo.csv")

X = data.drop(['type', 'name'], axis=1)                  # drop the target and the name from the features
y = data['type']                                         # set the target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)   # split the data into training and testing sets

## Entropy and Information Gain

Entropy, in information theory, is a measure of uncertainty or disorder within a set of outcomes. In the context of decision trees, it quantifies the impurity or homogeneity of a dataset. Lower entropy indicates a more pure subset, while higher entropy indicates greater uncertainty.

The entropy $ H $ of a random variable $ C $ can be mathematically defined as:

$$
H(C) = - \sum_{i} P(c_i) \log_2 P(c_i)
$$

where:
- $ H(C) $ is the entropy of the random variable $ C $,
- $ P(c_i) $ is the probability of outcome $ c_i $.

In decision trees, entropy helps determine how to split the data at each node by measuring the effectiveness of the splits in creating homogeneous subsets.


In [15]:
def entropy_from_distribution(prob_dist):
    """
    Computes the entropy of a probability distribution.
    
    Parameters:
        prob_dist (list or array): A list or array of probabilities.
        
    Returns:
        float: The entropy of the probability distribution.
    """
    # Your code goes here

    pass    






In [16]:
# solution

def entropy_from_distribution(prob_dist):
    """
    Computes the entropy of a probability distribution.
    
    Parameters:
        prob_dist (list or array): A list or array of probabilities.
        
    Returns:
        float: The entropy of the probability distribution.
    """
    prob_dist = np.array(prob_dist)
    # Ensure probabilities are non-zero to avoid log(0)
    prob_dist = prob_dist[prob_dist > 0]
    return -np.sum(prob_dist * np.log2(prob_dist))

In [ ]:
# Test the function
print(entropy_from_distribution([0.5, 0.5]))  # 1.0
print(entropy_from_distribution([0.2, 0.8]))  # 0.0
print(entropy_from_distribution([0.25, 0.75]))  # 0.8112781244591328
print(entropy_from_distribution([0.0, 1.0]))  # 0.0



In [ ]:
# Plot the entropy of a binary distribution
for p in range (1, 100, 1):
    p = p/100
    entropy = entropy_from_distribution([p, 1-p])
    plt.scatter(p, entropy, color='blue')
plt.xlabel('Probability of Success')
plt.ylabel('Entropy')
plt.title('Entropy vs. Probability of Success')
plt.grid(True)
plt.gca().set_aspect('equal', adjustable='box')   #make it square
plt.show()


Now, let's compute the entropy of a dataset. Our function `entropy` takes a vector _y_ (the target variable), counts the occurrences of each class, and uses the `compute_entropy` function defined above.

In [19]:
def entropy(y):
    """
    Computes the entropy of a list of class labels.
    
    Parameters:
        y (list or array): Class labels.
        
    Returns:
        float: The entropy of the class labels.
    """
    # Your code goes here
    pass









In [20]:
# solution

def entropy(y):
    """
    Computes the entropy of a list of class labels.
    
    Parameters:
        y (list or array): Class labels.
        
    Returns:
        float: The entropy of the class labels.
    """
    unique_labels = set(y)
    prob_dist = []
    for label in unique_labels:
        p = sum(y == label) / len(y)
        prob_dist.append(p) 
    return entropy_from_distribution(prob_dist)


It is now easy to compute the entropy of our entire training dataset.

In [ ]:
entropy(y_train)

The decision tree algoritm splits the dataset according to attribute values. 
It uses information gain to determine the best split at each node. 

Our implementation will keep track of the indexes of instances (examples) that are currently considered (instead of making copies of the dataset, which would be inefficient).

In [ ]:
X_train.head()

In [23]:


def split_dataset(X, node_indices, feature):
    """
    Splits the data into branches according to the feature.
    
    Args:
        X (pandas data frame):   The dataset
        node_indices (list):     List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        children_indices:        A dictionary containing the indices of the samples for each child node.
        
    """
    
    # Get distinct feature values for the given feature from the entire dataset (not just the active indices)
    feature_values = X[feature].unique()
    
    branches = {value: [] for value in feature_values}  # create a dictionary to hold the indices for each branch
    
    for i in node_indices:
        feature_value = X[feature] [i]  # get the feature value for the sample at index i
        branches[feature_value].append(i)
        
    return branches

In [ ]:
# Test the function
feature = "legs"

indices = X_train.index.values
print("All indices:", indices)
for k,v in split_dataset(X_train, indices, feature).items():
    print(k,v)



Compute the information gain for a given feature.

In [25]:
def information_gain(X, y, node_indices, feature):
    """
    Computes the information gain obtained by splitting the data at the given node using the given feature.
    
    Args:
        X (pandas dataframe):    Data matrix of shape
        y (pandas series):       Target labels
        node_indices (list):     List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        float: The information gain obtained by splitting the data on the given feature.
    """
    pass





In [26]:
# Solution

def information_gain(X, y, node_indices, feature):
    """
    Computes the information gain obtained by splitting the data at the given node using the given feature.
    
    Args:
        X (pandas dataframe):    Data matrix of shape
        y (pandas series):       Target labels
        node_indices (list):     List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        float: The information gain obtained by splitting the data on the given feature.
    """
    
    total_entropy = entropy(y[node_indices])            # compute the entropy of the current node
    branches = split_dataset(X, node_indices, feature)  # split the data into branches

    # compute the entropy of the children nodes
    children_entropy = 0
    for branch in branches.values():
        children_entropy += len(branch) / len(node_indices) * entropy(y[branch])   # weighted sum of entropies of the children nodes

    return total_entropy - children_entropy              # return the information gain

Selecting the best feature/attribute.

In [27]:
def get_best_split(X, y, node_indices):  
    """
    Returns the name of the best feature to split on.
    
    Returns:
        string: The name of the best feature to split on.
        float: The information gain obtained by splitting on the best feature.
    """
    
    best_feature = None
    best_gain = 0
    
    for feature in X.columns:
        gain = information_gain(X, y, node_indices, feature)
        if gain > best_gain:
            best_gain = gain
            best_feature = feature
            
    return best_feature, best_gain

In [ ]:
# Test the function
indices = X_train.index.values
get_best_split(X_train, y_train, indices)


We put it all together in a class.

In [29]:
class DecisionTree:
    """ This class implements a simple decision tree algorithm for classification tasks by using the helper functions defined above."""
    def __init__(self):
        self.tree = []

    def fit(self, X, y, node_name, node_indices, max_depth, current_depth):
        """
        Build a tree using the recursive algorithm that splits the dataset into subgroups according to attribute values.
        This function builds the tree and stores it in the 'self.tree' member attribute.
        
        Args:
            X (pandas dataframe):   Data matrix of shape (n_samples, n_features)
            y (array-like):         List or ndarray with n_samples containing the target variable.
            node_indices (ndarray): List containing the active indices, i.e., the samples being considered in this step.
            max_depth (int):        Max depth of the resulting tree. 
            current_depth (int):    Current depth. Parameter used during recursive call.
    
        Returns:
            None: Modifies the 'self.tree' variable in place.
        """
        
        # Get the best feature to split on at this node
        best_feature, best_gain = get_best_split(X, y, node_indices)


        # Maximum depth reached - stop splitting
        if current_depth == max_depth or best_gain == 0:   # the current node is a leaf
            self.tree.append({'type': 'leaf', 'depth': current_depth, 'node_name': node_name, 'class': y[node_indices].mode()[0]})
            return
        
        # Store the current split in the tree
        self.tree.append({'type': 'node', 'depth': current_depth, 'node_name': node_name})
        
        # Split the dataset at the best feature
        branches = split_dataset(X, node_indices, best_feature)
        
        # Continue building the tree recursively for the left and right subgroups
        for branch, indeces in branches.items():
            node_name = f"{best_feature} = {branch}"
            self.fit(X, y, node_name, indeces, max_depth, current_depth + 1)


    def visualize_tree(self):
        """
        Prints the decision tree stored self.tree as a list of dictionaries in an ASCII format.
        
        """
        
        for node in self.tree:
            # Create indentation based on the depth of the node
            indentation = " " * (node["depth"] * 4) + " " # 4 spaces for each level of depth
            
            if node['type'] == 'node':
                # Print decision node
                print(f"{indentation}{node['node_name']}")
            
            elif node['type'] == 'leaf':
                # Print leaf node with its class
                print(f"{indentation}{node['node_name']} --> Class: {node['class']}")

In [ ]:
# Test the function
tree = DecisionTree()
indices = X_train.index.values
tree.fit(X_train, y_train, "Root", indices, 2, 0)
tree.visualize_tree()

You can try it on other datasets as well, just make sure that the data is not numeric.

In [ ]:
# Load the Titanic dataset with pandas
from pathlib import Path

titanic_data = pd.read_csv(Path("../../data") / "titanic.csv")
# Split the data (vertically) into features (X) and target (y)
X = titanic_data[["age", "sex", "status"]]           # These are the independent variables (X is a DataFrame)
y = titanic_data['survived']                         # This is the target variable (y is a Series)


tree_titanic = DecisionTree()
titanic_indices = X.index.values
tree_titanic.fit(X, y, "Root", titanic_indices, 3, 0)
tree_titanic.visualize_tree()